# 00 · Concepts: context rot, RAG and RLM

This project asks one question: **when an LLM must read a huge amount of text, which strategy gives the right answer most cheaply?**
This notebook explains the three strategies we compare. No API calls; it runs offline.

## 1. Context rot
An LLM reads its whole prompt (the *context*) at once. As the prompt grows, accuracy drops, even well below the model's advertised window.
Details in the middle of a long prompt get "lost", and tasks that need *every* record (counting, comparing) suffer most.
This is **context rot**. Our benchmark measures it on contexts of 32K → 2M tokens.

| Strategy | What the model sees | Weak spot |
|---|---|---|
| **Direct LLM** | The entire context in one prompt | Context rot; it can't even run once the text exceeds the window |
| **RAG** | Only the top‑k chunks a retriever picks | Misses anything not retrieved, so it can't count over *all* records |
| **RLM** | A Python REPL holding the context as a variable | The model must write good code; cost varies a lot |

## 2. RAG in miniature
RAG = **R**etrieve the few most relevant chunks, then let the LLM **G**enerate an answer from them.
Below is a toy version: score each record by how many query words it contains and keep the top 2.

In [1]:
records = [
    "order A: delivered 7 days late, review 1/5 'produto chegou atrasado'",
    "order B: delivered early, review 5/5 'otimo produto'",
    "order C: delivered on time, review 2/5 'produto quebrado'",
    "order D: delivered 3 days late, review 2/5 'atrasou muito'",
]

def retrieve(query, docs, k=2):
    words = query.lower().split()
    return sorted(docs, key=lambda d: sum(w in d.lower() for w in words), reverse=True)[:k]

retrieve("why was order A late", records)

["order A: delivered 7 days late, review 1/5 'produto chegou atrasado'",
 "order D: delivered 3 days late, review 2/5 'atrasou muito'"]

That works for a **lookup** (about one specific order). Now ask *"how many orders were late?"*: the answer needs all 4 records,
but RAG only passes the top‑k to the LLM. With 100K orders and k=10, RAG can't count, **by design**.

## 3. RLM: Recursive Language Model
Idea from Zhang, Kraska & Khattab (MIT): **don't put the context in the prompt at all.**
1. The context is stored in a Python variable `context` inside a REPL.
2. The *root* LLM sees only the question plus a short description (e.g. "a string of 2M characters").
3. It writes code to look at the data: `context[:500]`, `context.split('\n')`, regex, loops.
4. For parts that need *understanding* (e.g. "does this review blame delivery?") it calls `llm_query(text)`, a **sub-LLM** run on a small slice.
5. When done, it returns `FINAL(answer)`.

*Recursive* = a sub-call can itself be an RLM (depth 2). Depth 0 = REPL only, no sub-calls.
Below is a **fake** RLM step, with code "written" by hand and a stand-in `llm_query`, just to show the shape of the loop.

In [2]:
context = "\n".join(records)          # in a real RLM this could be 2M tokens

def llm_query(text):                    # stand-in for a sub-LLM call
    return "delivery" if "atras" in text else "product"

# code the root model might write:
lines = context.split("\n")
late = [l for l in lines if "late" in l]
labels = [llm_query(l) for l in late]   # one cheap sub-call per record
print(len(late), "late orders →", labels)

2 late orders → ['delivery', 'delivery']


Each sub-call reads a tiny slice, so no single call suffers from context rot, yet together they cover **every** record.
The price: many calls, and quality depends on the code the model writes. Phase 4 builds this for real.

## 4. Our three question tiers
| Tier | Example | Work as data grows | Who should win |
|---|---|---|---|
| T1 Lookup | "Why was order X's customer unhappy?" | constant | RAG ≈ RLM |
| T2 Aggregation | "% of late-order comments blaming delivery?" | linear | RLM |
| T3 Pairwise | "Seller pairs in one state with opposite complaints?" | quadratic | only RLM |

Next: **01_data_exploration**, where we look at the real Olist data.